# Alpamayo 1.5: Local Offline Inference Demo

This notebook runs Alpamayo 1.5 on a local offline clip directory, shows all sampled images used for inference, visualizes the predicted trajectory and chain-of-thought, and supports sliding inference over the whole clip.

In [ ]:
import os
import sys
import time
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from transformers import BitsAndBytesConfig
from IPython.display import display, clear_output

from alpamayo1_5 import helper
from alpamayo1_5.load_offline_dataset import load_offline_dataset, _load_ego_pose_log
from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5

In [ ]:
# ===== Local paths =====
CLIP_DIR = Path('/workspace/dataset/')
MODEL_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Alpamayo-1.5-10B/snapshots/f11cd25b758ab560114019b555dde2a8b92d88b4')
PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b')
COSMOS_PROCESSOR_PATH = Path('/root/.cache/huggingface/hub/models--nvidia--Cosmos-Reason2-8B/snapshots/f715d875a8a99a0a2b65aa28633afd9417e46bd9')

# ===== Single inference config =====
DEVICE = 'cuda'
T0_SOD = 175490.23
NUM_HISTORY_STEPS = 16
NUM_FUTURE_STEPS = 64
TIME_STEP = 0.1
NUM_FRAMES = 4
FPS = 30.0
FRAME0_GPS_TIME_SOD = 175484.98
NUM_TRAJ_SAMPLES = 1
TEMPERATURE = 0.6
TOP_P = 0.98
MAX_GENERATION_LENGTH = 256

# ===== Sliding inference config =====
SLIDE_START_T0_SOD = 175490.23
SLIDE_END_T0_SOD = 175500.23
SLIDE_STEP_SOD = 5.0
SLIDE_CLEAR_OUTPUT_EACH_STEP = True

os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'] = str(COSMOS_PROCESSOR_PATH)

print('CLIP_DIR =', CLIP_DIR)
print('MODEL_PATH =', MODEL_PATH)
print('PROCESSOR_PATH =', PROCESSOR_PATH)
print('ALPAMAYO_VLM_PROCESSOR_PATH =', os.environ['ALPAMAYO_VLM_PROCESSOR_PATH'])
print('T0_SOD =', T0_SOD)
print('SLIDE_START_T0_SOD =', SLIDE_START_T0_SOD)
print('SLIDE_END_T0_SOD =', SLIDE_END_T0_SOD)
print('SLIDE_STEP_SOD =', SLIDE_STEP_SOD)

### Helper functions

In [ ]:
def wrap_to_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi


def global_motion_yaw_deg(xyz):
    xy = xyz[:, :2]
    disp = xy[-1] - xy[0]
    if np.linalg.norm(disp) < 1e-6:
        return np.nan
    return float(np.rad2deg(np.arctan2(disp[1], disp[0])))


def mean_speed_mps(xyz, dt):
    xy = xyz[:, :2]
    dxy = xy[1:] - xy[:-1]
    step_dist = np.linalg.norm(dxy, axis=1)
    if len(step_dist) == 0:
        return 0.0
    return float(step_dist.mean() / dt)


def compute_adaptive_xy_limits(hist_xyz, gt_xyz, pred_xyz_np, min_span=20.0, pad_ratio=0.12, pad_min=2.0):
    pts = [gt_xyz[:, :2], np.array([[0.0, 0.0]], dtype=np.float32)]
    if hist_xyz is not None:
        pts.append(hist_xyz[:, :2])
    for sample_idx in range(pred_xyz_np.shape[0]):
        pts.append(pred_xyz_np[sample_idx, :, :2])

    pts = np.concatenate(pts, axis=0)
    xmin, ymin = pts.min(axis=0)
    xmax, ymax = pts.max(axis=0)

    xspan = xmax - xmin
    yspan = ymax - ymin
    span = max(float(xspan), float(yspan), float(min_span))

    xcenter = 0.5 * (xmin + xmax)
    ycenter = 0.5 * (ymin + ymax)

    pad = max(span * pad_ratio, pad_min)
    half = 0.5 * span + pad

    xlim = (xcenter - half, xcenter + half)
    ylim = (ycenter - half, ycenter + half)
    return xlim, ylim


def plot_prediction_result(t0_sod, hist_xyz, gt_xyz, pred_xyz_np, min_ade, cot_text=None):
    xlim, ylim = compute_adaptive_xy_limits(hist_xyz, gt_xyz, pred_xyz_np)

    plt.figure(figsize=(6, 6))

    if hist_xyz is not None:
        plt.plot(hist_xyz[:, 0], hist_xyz[:, 1], 'o-', linewidth=2, markersize=3, alpha=0.9, label='History')

    plt.plot(gt_xyz[:, 0], gt_xyz[:, 1], 'k-o', linewidth=2, markersize=3, label='GT')

    for sample_idx in range(pred_xyz_np.shape[0]):
        plt.plot(
            pred_xyz_np[sample_idx, :, 0],
            pred_xyz_np[sample_idx, :, 1],
            '-o',
            linewidth=2,
            markersize=3,
            alpha=0.8,
            label=f'Pred {sample_idx}',
        )

    plt.scatter([0.0], [0.0], c='red', marker='x', s=100, label='t0 ego')
    plt.xlabel('x')
    plt.ylabel('y')
    title = f'Offline inference @ t0_sod={t0_sod:.2f}, minADE={min_ade:.3f}m'
    if cot_text is not None:
        cot_short = cot_text if len(cot_text) < 120 else cot_text[:117] + '...'
        title += f'\n{cot_short}'
    plt.title(title)
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.gca().set_aspect('equal', adjustable='box')
    plt.tight_layout()
    plt.show()


def run_single_inference_for_t0(t0_sod, model, processor, debug=False):
    data = load_offline_dataset(
        clip_dir=CLIP_DIR,
        t0_sod=float(t0_sod),
        num_history_steps=NUM_HISTORY_STEPS,
        num_future_steps=NUM_FUTURE_STEPS,
        time_step=TIME_STEP,
        num_frames=NUM_FRAMES,
        fps=FPS,
        frame0_gps_time_sod=FRAME0_GPS_TIME_SOD,
        debug=debug,
    )

    messages = helper.create_message(
        frames=data['image_frames'].flatten(0, 1),
        camera_indices=data['camera_indices'],
    )

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        continue_final_message=True,
        return_dict=True,
        return_tensors='pt',
    )

    model_inputs = {
        'tokenized_data': inputs,
        'ego_history_xyz': data['ego_history_xyz'],
        'ego_history_rot': data['ego_history_rot'],
    }
    model_inputs = helper.to_device(model_inputs, DEVICE)

    if DEVICE.startswith('cuda'):
        torch.cuda.manual_seed_all(42)
        autocast_context = torch.autocast('cuda', dtype=torch.bfloat16)
    else:
        autocast_context = torch.autocast(device_type=DEVICE, enabled=False)

    with autocast_context:
        pred_xyz, pred_rot, extra = model.sample_trajectories_from_data_with_vlm_rollout(
            data=model_inputs,
            top_p=TOP_P,
            temperature=TEMPERATURE,
            num_traj_samples=NUM_TRAJ_SAMPLES,
            max_generation_length=MAX_GENERATION_LENGTH,
            return_extra=True,
        )

    cot_value = extra['cot'][0]
    if hasattr(cot_value, 'tolist'):
        cot_value = cot_value.tolist()

    hist_xyz = data['ego_history_xyz'].cpu().numpy()[0, 0, :, :]
    gt_xyz = data['ego_future_xyz'].cpu().numpy()[0, 0, :, :]
    gt_xy = gt_xyz[:, :2].T

    pred_xyz_np = pred_xyz.cpu().numpy()[0, 0, :, :, :]
    pred_xy = pred_xyz_np[:, :, :2].transpose(0, 2, 1)

    diff = np.linalg.norm(pred_xy - gt_xy[None, ...], axis=1).mean(-1)
    min_ade = float(diff.min())
    best_idx = int(diff.argmin())

    pred_best_xyz = pred_xyz_np[best_idx]
    gt_final_xy = gt_xyz[-1, :2]
    pred_final_xy = pred_best_xyz[-1, :2]
    final_point_error = float(np.linalg.norm(pred_final_xy - gt_final_xy))

    gt_mean_speed = mean_speed_mps(gt_xyz, TIME_STEP)
    pred_mean_speed = mean_speed_mps(pred_best_xyz, TIME_STEP)
    speed_error = float(pred_mean_speed - gt_mean_speed)

    gt_yaw = global_motion_yaw_deg(gt_xyz)
    pred_yaw = global_motion_yaw_deg(pred_best_xyz)
    if np.isfinite(gt_yaw) and np.isfinite(pred_yaw):
        yaw_error = float(np.rad2deg(wrap_to_pi(np.deg2rad(pred_yaw - gt_yaw))))
    else:
        yaw_error = np.nan

    return {
        't0_sod': float(t0_sod),
        'data': data,
        'pred_xyz': pred_xyz,
        'pred_rot': pred_rot,
        'extra': extra,
        'cot_value': cot_value,
        'hist_xyz': hist_xyz,
        'gt_xyz': gt_xyz,
        'pred_xyz_np': pred_xyz_np,
        'min_ade': min_ade,
        'best_idx': best_idx,
        'final_point_error': final_point_error,
        'gt_mean_speed': gt_mean_speed,
        'pred_mean_speed': pred_mean_speed,
        'speed_error': speed_error,
        'gt_yaw': gt_yaw,
        'pred_yaw': pred_yaw,
        'yaw_error': yaw_error,
    }

### Load model and processor

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_enable_fp32_cpu_offload=False,
)

model = Alpamayo1_5.from_pretrained(
    str(MODEL_PATH),
    dtype=torch.bfloat16,
    device_map='cuda:0',
    quantization_config=quantization_config,
)

processor = helper.get_processor(
    model.tokenizer,
    processor_name_or_path=PROCESSOR_PATH,
)

print('Model and processor loaded.')

### Single inference

In [ ]:
result = run_single_inference_for_t0(
    t0_sod=T0_SOD,
    model=model,
    processor=processor,
    debug=True,
)

print('Chain-of-Causation:')
print(result['cot_value'])
print('minADE:', result['min_ade'], 'meters')
print('final_point_error:', result['final_point_error'], 'meters')
print('speed_error:', result['speed_error'], 'm/s')
print('yaw_error:', result['yaw_error'], 'deg')

plot_prediction_result(
    t0_sod=result['t0_sod'],
    hist_xyz=result['hist_xyz'],
    gt_xyz=result['gt_xyz'],
    pred_xyz_np=result['pred_xyz_np'],
    min_ade=result['min_ade'],
    cot_text=str(result['cot_value']),
)

### Sliding inference over the clip

In [ ]:
pose_times_sod, _, _ = _load_ego_pose_log(CLIP_DIR / 'ego_pos.log')

history_margin = (NUM_HISTORY_STEPS - 1) * TIME_STEP
future_margin = NUM_FUTURE_STEPS * TIME_STEP

valid_t0_min = float(pose_times_sod[0] + history_margin)
valid_t0_max = float(pose_times_sod[-1] - future_margin)

requested_t0_values = np.arange(SLIDE_START_T0_SOD, SLIDE_END_T0_SOD + 1e-9, SLIDE_STEP_SOD)
t0_values = requested_t0_values[
    (requested_t0_values >= valid_t0_min) & (requested_t0_values <= valid_t0_max)
]

print('pose_start =', float(pose_times_sod[0]))
print('pose_end   =', float(pose_times_sod[-1]))
print('valid_t0_min =', valid_t0_min)
print('valid_t0_max =', valid_t0_max)
print('requested_t0_values =', [round(float(x), 3) for x in requested_t0_values])
print('filtered_t0_values  =', [round(float(x), 3) for x in t0_values])

if len(t0_values) == 0:
    raise ValueError('No valid t0 values remain after filtering.')

slide_rows = []

for idx, t0_sod_i in enumerate(t0_values):
    t_start = time.time()
    
    try:
        result_i = run_single_inference_for_t0(
            t0_sod=float(t0_sod_i),
            model=model,
            processor=processor,
            debug=False,
        )

        elapsed = time.time() - t_start

        if SLIDE_CLEAR_OUTPUT_EACH_STEP:
            clear_output(wait=True)

        print(f'[{idx + 1}/{len(t0_values)}] t0_sod={t0_sod_i:.3f}')
        print('Chain-of-Causation:')
        print(result_i['cot_value'])
        print(
            f"minADE={result_i['min_ade']:.3f} m | "
            f"final_err={result_i['final_point_error']:.3f} m | "
            f"speed_err={result_i['speed_error']:.3f} m/s | "
            f"yaw_err={result_i['yaw_error']:.3f} deg | "
            f"time={elapsed:.2f}s"
        )

        plot_prediction_result(
            t0_sod=result_i['t0_sod'],
            hist_xyz=result_i['hist_xyz'],
            gt_xyz=result_i['gt_xyz'],
            pred_xyz_np=result_i['pred_xyz_np'],
            min_ade=result_i['min_ade'],
            cot_text=str(result_i['cot_value']),
        )

        slide_rows.append({
            't0_sod': float(t0_sod_i),
            'minADE_m': result_i['min_ade'],
            'best_sample_idx': result_i['best_idx'],
            'final_point_error_m': result_i['final_point_error'],
            'gt_mean_speed_mps': result_i['gt_mean_speed'],
            'pred_mean_speed_mps': result_i['pred_mean_speed'],
            'speed_error_mps': result_i['speed_error'],
            'gt_global_motion_yaw_deg': result_i['gt_yaw'],
            'pred_global_motion_yaw_deg': result_i['pred_yaw'],
            'yaw_error_deg': result_i['yaw_error'],
            'elapsed_sec': elapsed,
            'cot': str(result_i['cot_value']),
        })

    except Exception as e:
        elapsed = time.time() - t_start
        if SLIDE_CLEAR_OUTPUT_EACH_STEP:
            clear_output(wait=True)
        print(f'[{idx + 1}/{len(t0_values)}] t0_sod={t0_sod_i:.3f}')
        print('ERROR:', repr(e))
        slide_rows.append({
            't0_sod': float(t0_sod_i),
            'minADE_m': np.nan,
            'best_sample_idx': np.nan,
            'final_point_error_m': np.nan,
            'gt_mean_speed_mps': np.nan,
            'pred_mean_speed_mps': np.nan,
            'speed_error_mps': np.nan,
            'gt_global_motion_yaw_deg': np.nan,
            'pred_global_motion_yaw_deg': np.nan,
            'yaw_error_deg': np.nan,
            'elapsed_sec': elapsed,
            'cot': f'ERROR: {repr(e)}',
        })

df_slide = pd.DataFrame(slide_rows)
print('\nSliding summary:')
display(df_slide)

### Sliding summary stats

In [ ]:
summary_cols = [
    'minADE_m',
    'final_point_error_m',
    'speed_error_mps',
    'yaw_error_deg',
    'elapsed_sec',
]

display(df_slide[summary_cols].describe())

if df_slide['minADE_m'].notna().any():
    print('Best by minADE:')
    display(df_slide.loc[df_slide['minADE_m'].idxmin()].to_frame().T)

if df_slide['final_point_error_m'].notna().any():
    print('Best by final point error:')
    display(df_slide.loc[df_slide['final_point_error_m'].idxmin()].to_frame().T)